# Text Generation with an RNN (PyTorch)

A PyTorch port of the TensorFlow text generation tutorial. Trains a character-based GRU model on Shakespeare's writing from Andrej Karpathy's [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/). Given a sequence of characters the model learns to predict the next character, enabling autoregressive text generation.

**Note:** Enable GPU acceleration to execute this notebook faster. In Colab: *Runtime > Change runtime type > Hardware accelerator > GPU*.

## Setup

### Import PyTorch and other libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
import time
import requests

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Download the Shakespeare dataset

Change the path below to use your own data.

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path_to_file = 'shakespeare.txt'

if not os.path.exists(path_to_file):
    response = requests.get(url)
    with open(path_to_file, 'wb') as f:
        f.write(response.content)
    print(f'Downloaded {len(response.content):,} bytes to {path_to_file}')
else:
    print(f'File already exists: {path_to_file}')

### Read the data

First, look at the text:

In [ ]:
text = open(path_to_file, 'r', encoding='utf-8').read()
print(f'Length of text: {len(text)} characters')

In [ ]:
print(text[:250])

In [ ]:
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

## Process the text

### Vectorize the text

Instead of `tf.keras.layers.StringLookup` we build simple Python dicts mapping characters ↔ integer IDs.

In [ ]:
example_texts = ['abcdefg', 'xyz']

char2idx = {ch: idx for idx, ch in enumerate(vocab)}
idx2char = np.array(vocab)

for t in example_texts:
    ids = [char2idx[c] for c in t]
    print(f'{t!r} -> {ids}')

In [ ]:
# Invert: IDs back to characters
for t in example_texts:
    ids = [char2idx[c] for c in t]
    recovered = ''.join(idx2char[ids])
    print(recovered)

In [ ]:
def text_from_ids(ids):
    if isinstance(ids, torch.Tensor):
        ids = ids.cpu().numpy()
    return ''.join(idx2char[np.asarray(ids)])

### The prediction task

Given a sequence of characters, predict the next character at each time step.

### Create training examples and targets

Divide the text into non-overlapping chunks of `seq_length + 1` characters. Each chunk yields an `(input, target)` pair where `target` is `input` shifted one character to the right.

For example, with `seq_length = 4` and text `'Hello'`:
- input  → `'Hell'`
- target → `'ello'`

In [ ]:
all_ids = np.array([char2idx[c] for c in text], dtype=np.int64)
all_ids

In [ ]:
for i in all_ids[:10]:
    print(idx2char[i])

In [ ]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

# Pack into non-overlapping chunks of seq_length+1
num_chunks = len(all_ids) // (seq_length + 1)
data = all_ids[:num_chunks * (seq_length + 1)].reshape(num_chunks, seq_length + 1)

for seq in data[:5]:
    print(repr(''.join(idx2char[seq])))

In [ ]:
def split_input_target(sequence):
    input_text  = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

split_input_target(list('Tensorflow'))

In [ ]:
inp, tgt = split_input_target(data[0])
print('Input :', repr(text_from_ids(inp)))
print('Target:', repr(text_from_ids(tgt)))

### Create training batches

Use `torch.utils.data.Dataset` and `DataLoader` in place of `tf.data.Dataset`.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class ShakespeareDataset(Dataset):
    """Wraps the pre-chunked (N, seq_length+1) array."""
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chunk = self.data[idx]
        return chunk[:-1], chunk[1:]


BATCH_SIZE = 64

dataset_obj = ShakespeareDataset(data)
dataloader  = DataLoader(
    dataset_obj,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=(device.type == 'cuda'),  # faster host->GPU transfers
)

print(f'Dataset : {len(dataset_obj)} sequences')
print(f'Batches : {len(dataloader)} x {BATCH_SIZE}')

for inp_batch, tgt_batch in dataloader:
    print(f'Input  shape: {inp_batch.shape}')
    print(f'Target shape: {tgt_batch.shape}')
    break

## Build the Model

Three layers mirroring the Keras original:

| Keras | PyTorch |
|---|---|
| `Embedding` | `nn.Embedding` |
| `GRU` (return_sequences, return_state) | `nn.GRU(batch_first=True)` |
| `Dense` | `nn.Linear` |

In [ ]:
vocab_size    = len(vocab)
embedding_dim = 256
rnn_units     = 1024

In [ ]:
class MyModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.dense = nn.Linear(rnn_units, vocab_size)

    def forward(self, x, states=None):
        x = self.embedding(x)            # (batch, seq, embed_dim)
        x, states = self.gru(x, states)  # (batch, seq, rnn_units)
        x = self.dense(x)                # (batch, seq, vocab_size)
        return x, states

    def get_initial_state(self, batch_size):
        dev = next(self.parameters()).device
        return torch.zeros(1, batch_size, self.gru.hidden_size, device=dev)

In [ ]:
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
).to(device)

print(model)

For each character the model looks up the embedding, runs the GRU one time-step, and applies the linear layer to produce logits over the vocabulary.

## Try the model

Check output shapes and sample from the untrained model.

In [ ]:
for inp_batch, tgt_batch in dataloader:
    inp_batch = inp_batch.to(device)
    with torch.no_grad():
        example_batch_predictions, _ = model(inp_batch)
    print(example_batch_predictions.shape,
          '# (batch_size, sequence_length, vocab_size)')
    break

In [ ]:
# The model can run on inputs of any length
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}\n')
for name, param in model.named_parameters():
    print(f'  {name:30s} {str(list(param.shape)):25s} {param.numel():>10,} params')

In [ ]:
# Sample from the output distribution (not argmax, to avoid loops)
probs = torch.softmax(example_batch_predictions[0], dim=-1)  # (seq, vocab)
sampled_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)
sampled_ids = sampled_ids.cpu().numpy()

print('Input:\n', repr(text_from_ids(inp_batch[0].cpu())))
print()
print('Next Char Predictions:\n', repr(text_from_ids(sampled_ids)))

## Train the model

### Attach an optimizer and a loss function

`nn.CrossEntropyLoss` is the PyTorch equivalent of `SparseCategoricalCrossentropy(from_logits=True)`. It expects raw logits and integer class indices.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

example_batch_mean_loss = loss_fn(
    example_batch_predictions.reshape(-1, vocab_size),
    tgt_batch.to(device).reshape(-1),
)
print(f'Prediction shape : {example_batch_predictions.shape}'
      ' # (batch_size, sequence_length, vocab_size)')
print(f'Mean loss        : {example_batch_mean_loss.item():.6f}')

In [ ]:
import math
print(f'exp(mean loss) = {math.exp(example_batch_mean_loss.item()):.5f}')
print(f'(should be close to vocab_size = {vocab_size})')

### Configure checkpoints

In [ ]:
optimizer = torch.optim.Adam(model.parameters())

checkpoint_dir = './training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

### Execute the training

In [ ]:
EPOCHS = 20

In [ ]:
history = []

for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0.0

    for batch_n, (inp, target) in enumerate(dataloader):
        inp    = inp.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        predictions, _ = model(inp)

        # predictions : (batch, seq_len, vocab_size)
        # CrossEntropyLoss needs  (N, C) and (N,)
        loss = loss_fn(predictions.reshape(-1, vocab_size),
                       target.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    elapsed  = time.time() - start
    history.append(avg_loss)
    print(f'Epoch {epoch+1:2d}/{EPOCHS}  Loss: {avg_loss:.4f}  '
          f'Time: {elapsed:.2f}s')

    if (epoch + 1) % 5 == 0:
        ckpt = os.path.join(checkpoint_dir, f'ckpt_epoch_{epoch+1}.pt')
        torch.save(model.state_dict(), ckpt)
        print(f'  -> Saved checkpoint: {ckpt}')

## Generate text

Each call to `generate_one_step` feeds the current character(s) through the model and the GRU hidden state, then samples the next character from the output logits. The predicted character and updated state are passed back in on the next call.

The following `OneStep` class wraps the model for single-step generation:

In [ ]:
class OneStep(nn.Module):
    def __init__(self, model, char2idx, idx2char, temperature=1.0):
        super().__init__()
        self.model       = model
        self.char2idx    = char2idx
        self.idx2char    = idx2char
        self.temperature = temperature
        self._device     = next(model.parameters()).device

    def generate_one_step(self, inputs, states=None):
        """
        inputs : list of strings, one per batch element.
                 On the first call pass the full prompt; on subsequent calls
                 pass the single predicted character.
        states : GRU hidden state (1, batch, rnn_units) or None.
        returns: list of predicted chars (one per batch), updated states.
        """
        # Convert strings -> token ID tensors
        input_ids = torch.tensor(
            [[self.char2idx.get(c, 0) for c in s] for s in inputs],
            dtype=torch.long,
            device=self._device,
        )

        with torch.no_grad():
            predicted_logits, states = self.model(input_ids, states)

        # Take only the last time-step prediction
        predicted_logits = predicted_logits[:, -1, :]        # (batch, vocab)
        predicted_logits = predicted_logits / self.temperature

        # Sample from the distribution
        probs         = torch.softmax(predicted_logits, dim=-1)
        predicted_ids = torch.multinomial(probs, num_samples=1).squeeze(-1)

        predicted_chars = [self.idx2char[i.item()] for i in predicted_ids]
        return predicted_chars, states

In [ ]:
one_step_model = OneStep(model, char2idx, idx2char)

Run it in a loop to generate text. The model maintains its internal state across calls so it has full context.

In [ ]:
start      = time.time()
states     = None
next_chars = ['ROMEO:']
result     = ['ROMEO:']

for n in range(1000):
    next_chars, states = one_step_model.generate_one_step(next_chars,
                                                           states=states)
    result.append(next_chars[0])

output_text = ''.join(result)
print(output_text, '\n\n' + '_' * 80)
print(f'\nRun time: {time.time() - start:.3f}s')

If you want the model to generate text *faster*, batch the generation. Below the model generates 5 outputs in roughly the same time as 1.

In [ ]:
start      = time.time()
states     = None
next_chars = ['ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:', 'ROMEO:']
results    = [next_chars[:]]

for n in range(1000):
    next_chars, states = one_step_model.generate_one_step(next_chars,
                                                           states=states)
    results.append(next_chars[:])

# Reconstruct a full text string for each of the 5 sequences
num_seqs = len(results[0])
texts    = [''.join(step[j] for step in results) for j in range(num_seqs)]

for t in texts:
    print(t)
    print()
print(f'Run time: {time.time() - start:.3f}s')

## Export the generator

Save and restore the model with `torch.save` / `torch.load`.

In [ ]:
save_path = 'one_step_model.pt'
torch.save(model.state_dict(), save_path)
print(f'Model weights saved to {save_path}')

In [ ]:
# Reload
reloaded_base = MyModel(vocab_size, embedding_dim, rnn_units).to(device)
reloaded_base.load_state_dict(
    torch.load(save_path, map_location=device, weights_only=True)
)
reloaded_base.eval()
one_step_reloaded = OneStep(reloaded_base, char2idx, idx2char)
print('Reloaded successfully')

In [ ]:
states     = None
next_chars = ['ROMEO:']
result     = ['ROMEO:']

for n in range(100):
    next_chars, states = one_step_reloaded.generate_one_step(next_chars,
                                                              states=states)
    result.append(next_chars[0])

print(''.join(result))

## Advanced: Custom Training Loop

In the TensorFlow version this section showed how to override `train_step` with `tf.GradientTape`. In PyTorch *all* training is manual by default — the loop above already uses the explicit gradient approach. The cell below shows a slightly more detailed version with per-batch logging and periodic checkpointing, equivalent to the advanced section of the original notebook.

In [ ]:
# Re-initialise for a fresh training run
model2 = MyModel(vocab_size, embedding_dim, rnn_units).to(device)

optimizer2 = torch.optim.Adam(model2.parameters())
loss_fn2   = nn.CrossEntropyLoss()

EPOCHS2 = 10

for epoch in range(EPOCHS2):
    start        = time.time()
    running_loss = 0.0

    for batch_n, (inp, target) in enumerate(dataloader):
        inp    = inp.to(device)
        target = target.to(device)

        optimizer2.zero_grad()
        predictions, _ = model2(inp)
        loss = loss_fn2(predictions.reshape(-1, vocab_size),
                        target.reshape(-1))
        loss.backward()
        optimizer2.step()

        running_loss += loss.item()

        if batch_n % 50 == 0:
            print(f'Epoch {epoch+1} Batch {batch_n} Loss {loss.item():.4f}')

    avg = running_loss / len(dataloader)
    print(f'\nEpoch {epoch+1} Loss: {avg:.4f}  '
          f'Time: {time.time()-start:.2f}s')
    print('_' * 80)

    if (epoch + 1) % 5 == 0:
        ckpt = os.path.join(checkpoint_dir, f'ckpt2_epoch_{epoch+1}.pt')
        torch.save(model2.state_dict(), ckpt)

torch.save(model2.state_dict(),
           os.path.join(checkpoint_dir, 'ckpt2_final.pt'))